# GroundedRx — Bilingual Arabic-English Medical RAG

Multilingual (Arabic/English) medical Q&A over the PEACH patient-leaflet dataset.
(Was "FalconMed AI" -- renamed after the Falcon-H1 -> Qwen2.5 swap left the old name
referring to a model no longer in use.)
LangGraph retrieval pipeline (query rewrite -> embed -> retrieve -> quality-gate loop -> rerank) + Qwen2.5-7B-Instruct (4-bit NF4; was Falcon-H1-1.5B-Deep-Instruct) generation + BERTScore/DeepEval/LLM-as-judge evaluation (RAGAS dropped -- see CLAUDE.md).

**Before running:** Runtime -> Change runtime type -> GPU. Upload `qdrant_db_archive.zip` to `/content/` (Files panel, left sidebar), then run the cells below top to bottom.

**Component 7 (last cell)** is an optional Gradio demo UI -- run it after Components 4/5 to get a shareable public link (`*.gradio.live`) for live Q&A, no separate deployment needed. Link only lasts as long as this session does.


In [ ]:
# SETUP — Run this first after restart
# ponytail: PORTABLE across Colab and Kaggle -- auto-detected below, no manual edit
# needed. On Kaggle specifically, two things code can't do for you:
#   1. Notebook Settings -> Internet -> On (needed for pip install + model downloads)
#   2. Add Data -> Upload -> qdrant_db_archive.zip, attached as a Kaggle Dataset
#      (Kaggle mounts uploaded data read-only under /kaggle/input/; the notebook
#      unzips it into the writable /kaggle/working/ below, it can't unzip in place)
# UNVERIFIED: Kaggle's preinstalled package/CUDA stack has not been tested against
# this pip install list. If it conflicts, treat it the same as any other dependency
# issue in this notebook -- diagnose the actual conflict, don't guess-fix it (see
# the SGLang history in CLAUDE.md for what guess-fixing a stack conflict costs).
!pip install qdrant-client sentence-transformers langgraph langchain \
             langchain-community langchain-huggingface langdetect transformers \
             accelerate bitsandbytes bert-score rank_bm25 -q

# ponytail: the causal-conv1d/mamba-ssm install hint that used to live here is
# GONE, not forgotten -- it was specific to Falcon-H1's hybrid Mamba2 layers, which
# don't exist in Qwen2.5 (a standard attention-only transformer). See CLAUDE.md
# "Model swap" for why Falcon-H1 was replaced: two experiments (greedy decoding,
# 8-bit quantization) each failed to fix Arabic quality, converging on model
# capacity as the limitation rather than a decoding/quantization setting.

import glob, os, shutil

# ponytail: platform auto-detect. Colab has no /kaggle/input, so this is a safe,
# minimal check -- no env var or manual flag needed on either platform.
ON_KAGGLE = os.path.exists("/kaggle/input")
WORK_DIR  = "/kaggle/working" if ON_KAGGLE else "/content"
QDRANT_STORAGE = f"{WORK_DIR}/qdrant_storage"

if ON_KAGGLE:
    # BUG FIXED: originally searched for qdrant_db_archive.zip here, same as the
    # Colab path. That's wrong on Kaggle -- Kaggle auto-extracts any .zip uploaded
    # as a Dataset, so there is never a raw zip to unzip, only the already-extracted
    # store (meta.json + collection/ + .lock). Search for meta.json instead, the
    # store's own marker file.
    # Also: /kaggle/input/ is READ-ONLY, but Qdrant's local client writes a .lock
    # file into the store directory to claim it -- opening the store in place would
    # fail on a read-only mount regardless of the search fix above, so the extracted
    # store must be COPIED into the writable /kaggle/working/ first.
    _meta_matches = glob.glob("/kaggle/input/**/meta.json", recursive=True)
    assert _meta_matches, (
        "Qdrant store (meta.json) not found under /kaggle/input -- attach the "
        "qdrant_db_archive Dataset to this notebook first (Add Data -> search "
        "your dataset -> Add)."
    )
    _source_dir = os.path.dirname(_meta_matches[0])
    if not os.path.exists(QDRANT_STORAGE):
        shutil.copytree(_source_dir, QDRANT_STORAGE)
else:
    # Colab: raw zip uploaded to /content/, genuinely needs unzipping.
    ZIP_PATH = "/content/qdrant_db_archive.zip"
    !unzip "{ZIP_PATH}" -d "{QDRANT_STORAGE}" 2>/dev/null || echo "already extracted"

from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # reduces OOM from memory fragmentation across many small generate() calls

client      = QdrantClient(path=QDRANT_STORAGE)
embed_model = SentenceTransformer("BAAI/bge-m3", device="cuda")
reranker    = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda")

# bf16 has more dynamic range than fp16 under 4-bit quant -- a safe general default,
# not (unlike the old Falcon-H1 comment here) a fix for a specific overflow bug;
# Qwen2.5 is a standard attention transformer with no documented history of that.
# ponytail: falls back to fp16 only on GPUs without native bf16 (e.g. T4).
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# ponytail: EXPERIMENT #2 (8-bit quantization) was tested on Falcon-H1 and REJECTED --
# fixed the CJK-corruption symptom but 2/3 broader Arabic signals got worse and English
# unexpectedly regressed (see PROJECT_REPORT.md SS5.5/SS7 for the full numbers). Both
# experiments (decoding, then quantization) converged on "model capacity is the limit,"
# not a config setting -- hence the model swap below. Defaulting back to 4-bit NF4 (the
# original, best-understood baseline) so this model swap is the ONLY variable changed
# relative to the last full comparison, not stacked on top of an unresolved quantization
# question. Toggle preserved for a future A/B on the new model if it's ever needed.
USE_8BIT = False

bnb_config = (
    BitsAndBytesConfig(load_in_8bit=True)
    if USE_8BIT else
    BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4"
    )
)
# bitsandbytes' 8-bit kernel (MatMul8bitLt / LLM.int8()) only supports fp16 inputs
# internally -- loading in bf16 (when the GPU supports it) makes it silently cast
# bf16->fp16 on every single matmul call, correct but extremely noisy (one warning
# per layer per generated token) and a small repeated overhead. 4-bit NF4 has no such
# restriction, so this only changes the 8-bit path; the 4-bit baseline keeps compute_dtype.
model_dtype = torch.float16 if USE_8BIT else compute_dtype

# ponytail: MODEL SWAP -- was tiiuae/Falcon-H1-1.5B-Deep-Instruct. Two prior
# experiments (greedy decoding, 8-bit quantization) each failed to fix Arabic
# generation quality and converged on model capacity as the limitation, not a
# decoding/quantization setting -- see CLAUDE.md "Arabic generation quality
# investigation" for the full chain of evidence. Qwen2.5-7B-Instruct chosen over an
# Arabic-specialized model (e.g. Jais) as the first test: standard attention-only
# transformer architecture (avoids Falcon-H1's hybrid Mamba2 issues entirely --
# see the removed causal-conv1d/mamba-ssm hint above), mainstream transformers +
# bitsandbytes support (no SGLang-style dependency risk), and strong documented
# multilingual/Arabic benchmarks despite not being Arabic-specific. At 7B in 4-bit
# NF4, weights need ~4-5GB VRAM -- comfortable alongside bge-m3/reranker on a T4's
# ~14.5GB budget. trust_remote_code dropped: unlike Falcon-H1, Qwen2.5 has been in
# mainline transformers for a while and doesn't need it. UNVERIFIED: whether the HF
# repo is gated (would 401) has not been confirmed live -- if it 401s, that's the
# fix needed, not a code bug.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config, device_map="auto",
    dtype=model_dtype)
model.eval()

info = client.get_collection("peach_healthcare_multilingual")
print(f"Qdrant     : {info.points_count:,} vectors")
print(f"embed_model: loaded")
print(f"reranker   : loaded")
print(f"Model      : {MODEL_NAME} loaded (dtype: {model_dtype}, quantization: {'8-bit' if USE_8BIT else '4-bit NF4'})")
print(f"VRAM used    : {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"VRAM free    : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1024**3:.2f} GB")


In [ ]:
"""
================================================================
 Bilingual Arabic-English Medical RAG System
 Dataset : PEACH RAG Dataset (Patient Information Leaflets)
 Target  : ArabicNLP Workshop @ ACL/EMNLP
 Component 4 — LangGraph Query Processing Pipeline
================================================================
"""

# ════════════════════════════════════════════════════════════
# CELL 1 — Install (already done — skip if installed)
# ════════════════════════════════════════════════════════════

# ════════════════════════════════════════════════════════════
# CELL 2 — Imports
# ════════════════════════════════════════════════════════════
import logging
import torch
from typing import TypedDict, List, Optional
from langdetect import detect
from langgraph.graph import StateGraph, END
from qdrant_client.models import Filter, FieldCondition, MatchValue

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

# ════════════════════════════════════════════════════════════
# CELL 3 — Configuration
# ════════════════════════════════════════════════════════════
CONFIG_C4 = {
    "collection_name" : "peach_healthcare_multilingual",
    "reranker_model"  : "BAAI/bge-reranker-v2-m3",
    "top_k_retrieve"  : 20,
    "top_k_rerank"    : 5,
    "score_threshold" : 0.5,
    "max_rewrites"    : 2,
    "rrf_k"           : 60,   # Reciprocal Rank Fusion damping (standard default)
}

logger.info("Component 4 — LangGraph Query Pipeline")
logger.info(f"Collection      : {CONFIG_C4['collection_name']}")
logger.info(f"Retrieve top-k  : {CONFIG_C4['top_k_retrieve']}")
logger.info(f"Rerank top-k    : {CONFIG_C4['top_k_rerank']}")

# ════════════════════════════════════════════════════════════

# ════════════════════════════════════════════════════════════
# CELL 4b — BM25 sparse index (hybrid search)
# ════════════════════════════════════════════════════════════
# Dense embeddings fuzz exact tokens -- drug names, strengths, numbers. Measured
# on this corpus: BM25 puts "lisinopril dose 10 mg" on document 502 (the actual
# lisinopril leaflet) as its top hit, while bge-m3 alone spreads that across
# semantically-similar dosage text from unrelated drugs. BM25 is useless on
# generic phrasing ("side effects" scores ~7.9 and scatters), which is exactly
# where dense retrieval is strong. Complementary -> fuse, don't replace.
from rank_bm25 import BM25Okapi
import re

_pts, _ = client.scroll(
    collection_name = CONFIG_C4["collection_name"],
    limit           = 10000,
    with_payload    = True,
    with_vectors    = False,
)
BM25_DOCS = [
    {
        "id"         : p.id,
        "text"       : p.payload.get("chunk_text", ""),
        "language"   : p.payload.get("language", ""),
        "category"   : p.payload.get("category", ""),
        "document_id": p.payload.get("document_id", ""),
        "chunk_id"   : p.payload.get("chunk_id", ""),
        "file_name"  : p.payload.get("file_name", ""),
    }
    for p in _pts
]

def _tokenize(text: str) -> List[str]:
    # ponytail: unicode word split, no stemming or Arabic morphological analysis.
    # Verified to tokenize Arabic script correctly. Add a light Arabic stemmer only
    # if recall on inflected forms measurably suffers -- measure before adding.
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

bm25_index = BM25Okapi([_tokenize(d["text"]) for d in BM25_DOCS])
logger.info(f"BM25 index built over {len(BM25_DOCS):,} chunks")

# Known limitation: BM25 is lexical, so it cannot match a Latin-script drug name
# against an Arabic leaflet that spells it in Arabic script ("Logynon" scores 0.00
# -- the token simply isn't in the index). Cross-language matching stays the dense
# half's job; fusion is additive, so this costs nothing it wasn't already doing.


# ════════════════════════════════════════════════════════════
# CELL 5 — LangGraph State
# ════════════════════════════════════════════════════════════
class RAGState(TypedDict):
    query              : str
    language           : str
    rewritten_query    : str
    rewrite_count      : int
    query_vector       : List[float]
    retrieved_chunks   : List[dict]
    reranked_chunks    : List[dict]
    retrieval_score    : float
    context            : str
    needs_rewrite      : bool
    document_id_filter : Optional[int]  # eval-only: pin retrieval to one known document

# ════════════════════════════════════════════════════════════
# CELL 6 — Node definitions
# ════════════════════════════════════════════════════════════

# ── Node 1: Language Detection ──
def detect_language(state: RAGState) -> RAGState:
    """
    Detect query language using langdetect.
    Maps to 'ar' or 'en' for prompt template selection.
    """
    try:
        lang     = detect(state["query"])
        language = "ar" if lang == "ar" else "en"
    except Exception:
        language = "en"
    logger.info(f"Language: {language.upper()} | Query: {state['query'][:60]}")
    return {**state, "language": language}


# ── Node 2: Query Rewriting ──
def rewrite_query(state: RAGState) -> RAGState:
    """
    Pass 0 embeds the query as-is; retry passes expand with medical domain terms.
    """
    query         = state["query"]
    language      = state["language"]
    rewrite_count = state.get("rewrite_count", 0)

    # ponytail: keyword expansion is a RETRY lever, not a default transform.
    # Component 4b measured that appending language-specific medical keywords
    # halves cross-lingual retrieval for Arabic queries (60% -> 33% of top-20
    # crossing the language boundary) by pulling the embedding back toward the
    # query's own language; English queries are unaffected. Raw queries already
    # clear the 0.5 quality gate (cosine 0.63-0.73), so expansion only earns its
    # cost once the gate has actually failed.
    # This also fixes a dead feedback loop: expansion is built from state["query"]
    # (always the raw query), so both passes previously produced a byte-identical
    # string -- the retry re-ran identical retrieval and could never change anything.
    if rewrite_count == 0:
        logger.info(f"Pass 0 [{language.upper()}]: raw query, no expansion")
        return {**state, "rewritten_query": query, "rewrite_count": 1}

    if language == "ar":
        rewritten = (
            f"{query} "
            f"معلومات دوائية آثار جانبية جرعة تحذيرات "
            f"نشرة المريض تخزين الدواء"
        )
    else:
        rewritten = (
            f"{query} "
            f"medication information side effects dosage "
            f"warnings patient leaflet storage instructions"
        )

    logger.info(f"Rewritten [{language.upper()}]: {rewritten[:100]}")
    return {
        **state,
        "rewritten_query": rewritten,
        "rewrite_count"  : rewrite_count + 1,
    }


# ── Node 3: Query Embedding ──
def embed_query(state: RAGState) -> RAGState:
    """
    Embed rewritten query using bge-m3.
    Same model + normalization as indexing step.
    """
    query  = state.get("rewritten_query", state["query"])
    vector = embed_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()
    logger.info(f"Query embedded | dim: {len(vector)}")
    return {**state, "query_vector": vector}


# ── Node 4: Retrieval ──
def retrieve_chunks(state: RAGState) -> RAGState:
    """
    Hybrid retrieval: dense (bge-m3 / Qdrant cosine) + sparse (BM25),
    combined with Reciprocal Rank Fusion.
    """
    # ponytail: eval-only filter -- generic EVAL_QA questions ("this medication")
    # would otherwise match whichever of the 464 different drugs in the corpus is
    # semantically closest, producing inconsistent cross-question/cross-language
    # grounding. Live queries (document_id_filter unset) search the whole corpus
    # as normal -- this never touches real user-facing retrieval behavior.
    doc_filter = state.get("document_id_filter")
    query_filter = (
        Filter(must=[FieldCondition(key="document_id", match=MatchValue(value=doc_filter))])
        if doc_filter is not None else None
    )

    k     = CONFIG_C4["top_k_retrieve"]
    query = state.get("rewritten_query", state["query"])

    # ── dense half ──
    dense_hits = client.query_points(
        collection_name = CONFIG_C4["collection_name"],
        query            = state["query_vector"],
        limit             = k,
        query_filter       = query_filter,
    ).points

    # ── sparse half ──
    bm_scores = bm25_index.get_scores(_tokenize(query))
    cand_idx  = range(len(BM25_DOCS))
    if doc_filter is not None:
        cand_idx = [i for i in cand_idx if BM25_DOCS[i]["document_id"] == doc_filter]
    sparse_idx = sorted(cand_idx, key=lambda i: bm_scores[i], reverse=True)[:k]

    # ── Reciprocal Rank Fusion ──
    # Fuse on RANK, not raw score: cosine is bounded 0-1 while BM25 is unbounded and
    # its scale shifts per query, so the two are not directly comparable and any
    # score-weighted blend would need per-query normalization. RRF sidesteps that.
    # Point id is the join key -- verified unique across all 2,365 chunks.
    rrf_k, fused, meta = CONFIG_C4["rrf_k"], {}, {}

    for rank, h in enumerate(dense_hits):
        fused[h.id] = fused.get(h.id, 0.0) + 1.0 / (rrf_k + rank + 1)
        meta[h.id] = {
            "text"       : h.payload.get("chunk_text", ""),
            "language"   : h.payload.get("language", ""),
            "category"   : h.payload.get("category", ""),
            "document_id": h.payload.get("document_id", ""),
            "chunk_id"   : h.payload.get("chunk_id", ""),
            "file_name"  : h.payload.get("file_name", ""),
            "score"      : h.score,   # dense cosine
            "bm25_score" : 0.0,
        }

    for rank, i in enumerate(sparse_idx):
        d = BM25_DOCS[i]
        fused[d["id"]] = fused.get(d["id"], 0.0) + 1.0 / (rrf_k + rank + 1)
        # sparse-only hit: no cosine was ever computed for it, so score stays 0.0
        meta.setdefault(d["id"], {**{x: d[x] for x in
            ("text", "language", "category", "document_id", "chunk_id", "file_name")},
            "score": 0.0})
        meta[d["id"]]["bm25_score"] = float(bm_scores[i])

    ranked = sorted(fused.items(), key=lambda kv: kv[1], reverse=True)[:k]
    chunks = [{**meta[pid], "rrf_score": s} for pid, s in ranked]

    # retrieval_score stays the best DENSE cosine, deliberately. The quality gate
    # below is calibrated against cosine (threshold 0.5); feeding it an RRF score
    # (max ~0.016) or a raw BM25 score (unbounded) would make the gate fire on
    # every query or on none. Fusion changes what we retrieve, not how we judge it.
    best_score    = dense_hits[0].score if dense_hits else 0.0
    n_sparse_only = sum(1 for ch in chunks if ch["score"] == 0.0)
    logger.info(
        f"Hybrid retrieved {len(chunks)} chunks "
        f"({n_sparse_only} sparse-only) | Best dense: {best_score:.4f}"
    )

    return {
        **state,
        "retrieved_chunks": chunks,
        "retrieval_score" : best_score,
    }


# ── Node 5: Quality Check ──
def check_retrieval_quality(state: RAGState) -> RAGState:
    """
    If score < threshold AND rewrites remaining → rewrite.
    Implements LangGraph feedback loop.
    """
    score         = state["retrieval_score"]
    rewrite_count = state.get("rewrite_count", 0)
    needs_rewrite = (
        score < CONFIG_C4["score_threshold"] and
        rewrite_count < CONFIG_C4["max_rewrites"]
    )

    if needs_rewrite:
        logger.warning(
            f"Low score: {score:.4f} | "
            f"Rewrite {rewrite_count}/{CONFIG_C4['max_rewrites']}"
        )
    else:
        logger.info(f"Quality OK: {score:.4f}")

    return {**state, "needs_rewrite": needs_rewrite}


# ── Node 6: Reranking ──
def rerank_chunks(state: RAGState) -> RAGState:
    """
    CrossEncoder reranking: top-20 → top-5.
    More accurate than cosine similarity alone.
    """
    query  = state.get("rewritten_query", state["query"])
    chunks = state["retrieved_chunks"]

    if not chunks:
        return {**state, "reranked_chunks": []}

    pairs  = [(query, c["text"]) for c in chunks]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, chunks),
        key=lambda x: x[0],
        reverse=True
    )

    top_chunks = [
        {**chunk, "rerank_score": float(score)}
        for score, chunk in ranked[:CONFIG_C4["top_k_rerank"]]
    ]

    logger.info(
        f"Reranked {len(chunks)} → {len(top_chunks)} | "
        f"Top: {top_chunks[0]['rerank_score']:.4f}"
    )
    return {**state, "reranked_chunks": top_chunks}


# ── Node 7: Context Builder ──
def build_context(state: RAGState) -> RAGState:
    """
    Merge top-5 chunks into structured context for the LLM (MODEL_NAME, set in Setup).
    """
    chunks   = state["reranked_chunks"]
    language = state["language"]

    if not chunks:
        context = "No relevant information found." if language == "en" \
                  else "لم يتم العثور على معلومات ذات صلة."
        return {**state, "context": context}

    parts = []
    for i, chunk in enumerate(chunks, 1):
        header = f"[Source {i}]" if language == "en" else f"[المصدر {i}]"
        parts.append(
            f"{header}\n"
            f"Category : {chunk.get('category', 'N/A')}\n"
            f"Language : {chunk.get('language', 'N/A')}\n"
            f"Text     : {chunk['text']}\n"
        )

    context = "\n---\n".join(parts)
    logger.info(f"Context built | {len(chunks)} chunks | {len(context):,} chars")
    return {**state, "context": context}


# ════════════════════════════════════════════════════════════
# CELL 7 — Routing function
# ════════════════════════════════════════════════════════════
def route_after_quality_check(state: RAGState) -> str:
    if state.get("needs_rewrite", False):
        return "rewrite_query"
    return "rerank_chunks"


# ════════════════════════════════════════════════════════════
# CELL 8 — Build LangGraph pipeline
# ════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("Building LangGraph Pipeline")
logger.info("=" * 60)

workflow = StateGraph(RAGState)

workflow.add_node("detect_language",         detect_language)
workflow.add_node("rewrite_query",           rewrite_query)
workflow.add_node("embed_query",             embed_query)
workflow.add_node("retrieve_chunks",         retrieve_chunks)
workflow.add_node("check_retrieval_quality", check_retrieval_quality)
workflow.add_node("rerank_chunks",           rerank_chunks)
workflow.add_node("build_context",           build_context)

workflow.set_entry_point("detect_language")
workflow.add_edge("detect_language",         "rewrite_query")
workflow.add_edge("rewrite_query",           "embed_query")
workflow.add_edge("embed_query",             "retrieve_chunks")
workflow.add_edge("retrieve_chunks",         "check_retrieval_quality")

workflow.add_conditional_edges(
    "check_retrieval_quality",
    route_after_quality_check,
    {
        "rewrite_query": "rewrite_query",
        "rerank_chunks": "rerank_chunks",
    }
)

workflow.add_edge("rerank_chunks", "build_context")
workflow.add_edge("build_context", END)

rag_pipeline = workflow.compile()
logger.info("✅ LangGraph pipeline compiled")

# ════════════════════════════════════════════════════════════
# CELL 9 — Pipeline runner
# ════════════════════════════════════════════════════════════
def run_pipeline(query: str, document_id_filter: Optional[int] = None) -> dict:
    initial_state = RAGState(
        query              = query,
        language           = "",
        rewritten_query    = "",
        rewrite_count      = 0,
        query_vector       = [],
        retrieved_chunks   = [],
        reranked_chunks    = [],
        retrieval_score    = 0.0,
        context            = "",
        needs_rewrite      = False,
        document_id_filter = document_id_filter,
    )
    return rag_pipeline.invoke(initial_state)

# ════════════════════════════════════════════════════════════
# CELL 10 — Tests
# ════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("Testing Component 4")
logger.info("=" * 60)

# ── Test 1: English ──
print("\n" + "=" * 60)
print("TEST 1 — English: Side Effects")
print("=" * 60)
r1 = run_pipeline("What are the side effects of this medication?")
print(f"Query          : {r1['query']}")
print(f"Language       : {r1['language'].upper()}")
print(f"Retrieved      : {len(r1['retrieved_chunks'])} chunks")
print(f"Reranked       : {len(r1['reranked_chunks'])} chunks")
print(f"Retrieval score: {r1['retrieval_score']:.4f}")
print(f"\n── Context Preview ──\n{r1['context'][:500]}")

# ── Test 2: Arabic ──
print("\n" + "=" * 60)
print("TEST 2 — Arabic: Side Effects")
print("=" * 60)
r2 = run_pipeline("ما هي الآثار الجانبية لهذا الدواء؟")
print(f"Query          : {r2['query']}")
print(f"Language       : {r2['language'].upper()}")
print(f"Retrieved      : {len(r2['retrieved_chunks'])} chunks")
print(f"Reranked       : {len(r2['reranked_chunks'])} chunks")
print(f"Retrieval score: {r2['retrieval_score']:.4f}")
print(f"\n── Context Preview ──\n{r2['context'][:500]}")

# ── Test 3: Dosage ──
print("\n" + "=" * 60)
print("TEST 3 — English: Dosage")
print("=" * 60)
r3 = run_pipeline("What is the recommended dosage for adults?")
print(f"Query          : {r3['query']}")
print(f"Retrieval score: {r3['retrieval_score']:.4f}")
print(f"\n── Context Preview ──\n{r3['context'][:400]}")

# ── Test 4: Arabic storage ──
print("\n" + "=" * 60)
print("TEST 4 — Arabic: Storage")
print("=" * 60)
r4 = run_pipeline("كيف يتم تخزين هذا الدواء؟")
print(f"Query          : {r4['query']}")
print(f"Retrieval score: {r4['retrieval_score']:.4f}")
print(f"\n── Context Preview ──\n{r4['context'][:400]}")

# ── Test 5: Hybrid search — exact-term query ──
# Tests 1-4 are all generic phrasing, which is precisely where BM25 contributes
# nothing (measured on this corpus: "side effects" scores ~7.9 and scatters across
# unrelated leaflets). Identical results before/after fusion on those queries is
# the EXPECTED outcome, not a regression. Exact tokens -- drug names, strengths,
# numbers -- are where the sparse half earns its place, so test that explicitly
# or hybrid search is unfalsifiable from the smoke tests alone.
print("\n" + "=" * 60)
print("TEST 5 — Hybrid: exact-term (drug name + strength)")
print("=" * 60)
r5 = run_pipeline("What is the lisinopril 10 mg dose?")
n_sparse_only = sum(1 for ch in r5["retrieved_chunks"] if ch["score"] == 0.0)
n_both        = sum(1 for ch in r5["retrieved_chunks"]
                    if ch["score"] > 0.0 and ch.get("bm25_score", 0.0) > 0.0)
print(f"Query            : {r5['query']}")
print(f"Retrieved        : {len(r5['retrieved_chunks'])} chunks")
print(f"  BM25-only      : {n_sparse_only}  <- chunks dense retrieval never surfaced")
print(f"  found by both  : {n_both}")
print(f"Docs in top-5    : {sorted({ch['document_id'] for ch in r5['reranked_chunks']})}")
print(f"Retrieval score  : {r5['retrieval_score']:.4f}  (dense cosine, unaffected by fusion)")
print(f"\n── Context Preview ──\n{r5['context'][:400]}")

if n_sparse_only == 0:
    print("\n[!] BM25 contributed nothing here -- dense already covered its top-20.")
    print("    Lower rrf_k in CONFIG_C4 to weight sparse ranks more heavily.")
else:
    print(f"\n[OK] Fusion added {n_sparse_only} chunks dense retrieval missed entirely.")


# ════════════════════════════════════════════════════════════
# CELL 11 — Summary
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  COMPONENT 4 COMPLETE — QUERY PIPELINE READY")
print("=" * 60)
print("  ✅ Language Detection   — langdetect")
print("  ✅ Query Rewriting      — raw pass 0, expansion on retry only")
print("  ✅ Query Embedding      — BAAI/bge-m3 (1024-dim)")
print("  ✅ Hybrid Retrieval     — dense (bge-m3) + BM25, RRF-fused, top-20")
print("  ✅ Quality Check        — feedback loop if score < 0.5")
print("  ✅ Reranking            — bge-reranker-v2-m3 top-20→5")
print("  ✅ Context Builder      — structured medical context")
print("=" * 60)
print("\n✅ Component 4 Complete")
print(f"   Ready for Component 5 — {MODEL_NAME} Generation")

In [ ]:
"""
================================================================
 Component 4b — Cross-Lingual Retrieval Verification
 Does an Arabic query actually retrieve English chunks (and vice versa)?
 Runs on the retrieval stack only (Qdrant + bge-m3 + reranker) — no Falcon,
 no LLM judge. Seconds, not minutes.
================================================================
"""

# The corpus has 464 documents, each in exactly ONE language (no parallel
# AR/EN document pairs). So "cross-lingual retrieval" can only mean: a query
# in one language surfaces chunks written in the other. This cell tests that
# claim directly instead of assuming bge-m3's multilingual training delivers it.

LANG_FIELD = {"en": "English", "ar": "Arabic"}


def _retrieve(vector, lang_value=None, limit=20):
    """Raw Qdrant hit list, optionally pinned to one payload language."""
    qf = (
        Filter(must=[FieldCondition(key="language", match=MatchValue(value=lang_value))])
        if lang_value else None
    )
    return client.query_points(
        collection_name=CONFIG_C4["collection_name"],
        query=vector,
        limit=limit,
        query_filter=qf,
    ).points


# Semantically parallel EN/AR probes — same medical intent, different language.
# Not translations of one document; they only need to mean the same thing.
PROBE_PAIRS = [
    ("What are the side effects of this medicine?", "ما هي الآثار الجانبية لهذا الدواء؟"),
    ("How should this medicine be stored?",         "كيف يتم تخزين هذا الدواء؟"),
    ("What is the recommended dose for adults?",     "ما هي الجرعة الموصى بها للبالغين؟"),
    ("When should I not take this medicine?",        "متى يجب ألا أتناول هذا الدواء؟"),
]

print("=" * 70)
print("  CROSS-LINGUAL RETRIEVAL VERIFICATION")
print("=" * 70)

xling_rows = []

for en_q, ar_q in PROBE_PAIRS:
    for lang, query in (("en", en_q), ("ar", ar_q)):
        other = "ar" if lang == "en" else "en"

        # TEST A — unfiltered retrieval: what language mix comes back naturally?
        # Two variants, because rewrite_query appends language-specific medical
        # keywords, which is a prime suspect for pinning results to one language.
        raw_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
        # rewrite_count=1 -> the RETRY path. Pass 0 now returns the query unchanged,
        # so probing it would just re-measure the raw query and prove nothing.
        rew_state = rewrite_query({"query": query, "language": lang, "rewrite_count": 1})
        rew_vec = embed_model.encode(
            rew_state["rewritten_query"], normalize_embeddings=True
        ).tolist()

        def other_share(vec):
            hits = _retrieve(vec)
            n_other = sum(
                1 for h in hits if h.payload.get("language") == LANG_FIELD[other]
            )
            return n_other / len(hits) if hits else 0.0

        share_raw = other_share(raw_vec)
        share_rew = other_share(rew_vec)

        # TEST B — forced cross-language: pin retrieval to the OTHER language and
        # compare best score against same-language best. This isolates "can bge-m3
        # align across languages at all" from "does the corpus happen to favour
        # same-language chunks". If the two scores are close, alignment works and
        # TEST A's mix is just competition, not incapability.
        same_hits = _retrieve(raw_vec, LANG_FIELD[lang], limit=5)
        cross_hits = _retrieve(raw_vec, LANG_FIELD[other], limit=5)
        same_best = same_hits[0].score if same_hits else 0.0
        cross_best = cross_hits[0].score if cross_hits else 0.0

        # TEST C — second opinion from the multilingual reranker. Qdrant cosine and
        # the CrossEncoder can disagree; if the reranker also scores the cross-language
        # chunk highly, the match is real and not an embedding-space artifact.
        cross_rr = (
            float(reranker.predict([(query, cross_hits[0].payload.get("chunk_text", ""))])[0])
            if cross_hits else 0.0
        )
        same_rr = (
            float(reranker.predict([(query, same_hits[0].payload.get("chunk_text", ""))])[0])
            if same_hits else 0.0
        )

        xling_rows.append({
            "query": query,
            "query_lang": lang,
            "other_lang_share_raw": share_raw,
            "other_lang_share_rewritten": share_rew,
            "same_lang_best_score": same_best,
            "cross_lang_best_score": cross_best,
            "same_lang_rerank": same_rr,
            "cross_lang_rerank": cross_rr,
        })

        print(f"\n[{lang.upper()}] {query[:55]}")
        print(f"  A. {other.upper()} chunks in unfiltered top-20 : "
              f"raw {share_raw:.0%} | after rewrite {share_rew:.0%}")
        print(f"  B. best cosine  same-lang {same_best:.4f} | cross-lang {cross_best:.4f}")
        print(f"  C. best rerank  same-lang {same_rr:+.3f} | cross-lang {cross_rr:+.3f}")
        if cross_hits:
            print(f"     top {other.upper()} chunk: "
                  f"{cross_hits[0].payload.get('chunk_text','')[:90]}")

print("\n" + "=" * 70)
print("  VERDICT")
print("=" * 70)

mean = lambda k: sum(r[k] for r in xling_rows) / len(xling_rows)
mean_raw = mean("other_lang_share_raw")
mean_rew = mean("other_lang_share_rewritten")
score_gap = mean("same_lang_best_score") - mean("cross_lang_best_score")
rr_cross = mean("cross_lang_rerank")

print(f"Mean other-language share, raw query       : {mean_raw:.1%}")
print(f"Mean other-language share, rewritten query : {mean_rew:.1%}")
print(f"Mean cosine gap (same - cross)             : {score_gap:+.4f}")
print(f"Mean cross-language rerank score           : {rr_cross:+.3f}")
print()

if mean_raw < 0.05:
    print("A: Cross-lingual retrieval is NOT happening in practice — the pipeline")
    print("   returns same-language chunks almost exclusively.")
else:
    print(f"A: Cross-lingual retrieval IS happening — {mean_raw:.0%} of top-20 hits")
    print("   cross the language boundary unprompted.")

if mean_rew < mean_raw - 0.02:
    print("   ...and rewrite_query makes it WORSE: appending language-specific")
    print("   medical keywords pushes the embedding toward the query's own language.")

if score_gap < 0.10:
    print("B: bge-m3's cross-lingual alignment is sound — forced cross-language")
    print("   retrieval scores nearly as high as same-language.")
else:
    print("B: Large same-vs-cross score gap — cross-language matches are genuinely")
    print("   weaker in this embedding space, not just out-competed.")

if rr_cross > 0:
    print("C: The reranker agrees the cross-language chunks are relevant.")
else:
    print("C: The reranker rejects the cross-language chunks — they are topically")
    print("   off even when the embedding puts them close.")

# ponytail: one runnable check -- fails loudly if the probe itself broke
# (empty corpus, wrong language field values), rather than silently reporting 0%.
assert min(r["same_lang_best_score"] for r in xling_rows) > 0, \
    "same-language retrieval returned nothing -- check LANG_FIELD values match the payload"


In [ ]:
"""
================================================================
 Component 5 — LLM Generation (Qwen2.5-7B-Instruct; was Falcon-H1-1.5B-Deep-Instruct)
================================================================
"""

# ════════════════════════════════════════════════════════════
# CELL 1 — Prompt templates
# ════════════════════════════════════════════════════════════
PROMPT_EN = """You are a bilingual medical assistant specializing in patient information leaflets.
Answer the question using ONLY the information provided in the context below.
If the answer is not found in the context, say exactly: "I don't have enough information to answer this question."
Do NOT add any medical information not present in the context.

Context:
{context}

Question: {query}

Answer:"""

PROMPT_AR = """أنت مساعد طبي متخصص في نشرات معلومات المرضى.
أجب على السؤال باستخدام المعلومات الواردة في السياق أدناه فقط.
إذا لم تكن الإجابة موجودة في السياق، قل بالضبط: "لا أملك معلومات كافية للإجابة على هذا السؤال."
لا تضف أي معلومات طبية غير موجودة في السياق.

السياق:
{context}

السؤال: {query}

الإجابة:"""

print("✅ Prompt templates defined")

# ════════════════════════════════════════════════════════════
# CELL 2 — Generation function
# ════════════════════════════════════════════════════════════
import torch

def generate_answer(query: str, context: str, language: str) -> dict:
    """
    Generate medical answer using the loaded model (MODEL_NAME, set in Setup).
    Uses context-only prompting — prevents hallucination.
    Separate AR/EN prompt templates for better response quality.
    """
    template          = PROMPT_AR if language == "ar" else PROMPT_EN
    context_truncated = context[:2000]  # safe VRAM limit

    prompt = template.format(context=context_truncated, query=query)

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt = True,
        tokenize               = True,
        return_dict             = True,
        return_tensors           = "pt",
        truncation                = True,
        max_length                 = 2048,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]

    # ponytail: EXPERIMENT, not a confirmed fix. Leading hypothesis is that sampling
    # under 4-bit NF4 quantization draws noisy low-probability tokens more often for
    # Arabic (lower-frequency in training) than English -- observed once as a Chinese
    # character embedded mid-Arabic answer. Greedy removes that risk outright rather
    # than narrowing it, and matches what Component 6's judge/DeepEval calls already
    # do. Do NOT treat this as "fixed" until the Component 6 rerun + 20-answer manual
    # audit (see PROJECT_REPORT.md) actually show Arabic quality moved. If it doesn't
    # move, decoding was not the bottleneck -- see PROJECT_REPORT.md's precision/model
    # decision tree before touching quantization or the model.
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            # BUG FIXED: 300 was tuned for Falcon-H1's typically-shorter answers.
            # Measured live under Qwen2.5-7B: a real Arabic side-effects answer hit
            # the 300 cap and truncated mid-word, mid-bullet-point. Qwen produces
            # longer, more structured answers (severity-tiered bullet lists) than
            # Falcon-H1 ever did -- 450 gives real headroom based on what was
            # actually needed, not a guess.
            max_new_tokens     = 450,
            do_sample          = False,   # greedy -- deterministic, matches Component 6's judge calls
            repetition_penalty = 1.1,     # kept: greedy without it can loop/repeat; unrelated to sampling noise
            pad_token_id       = tokenizer.eos_token_id,
            eos_token_id       = tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_len:]
    answer     = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return {
        "answer"       : answer,
        "language"     : language,
        "input_tokens" : input_len,
        "output_tokens": len(new_tokens),
        # the TRUNCATED context -- the gate must judge grounding against what the
        # model was actually shown, not the full context it never received.
        "context_used" : context_truncated,
    }

print("✅ Generation function defined")

# ════════════════════════════════════════════════════════════
# CELL 2b — Runtime groundedness gate
# ════════════════════════════════════════════════════════════
# Deliberately NOT an LLM self-judge. DeepEval's FaithfulnessMetric failed on every
# question with the 1.5B model this was originally built against (its verdict JSON
# was unparseable regardless of token budget) -- a second generation call per query
# is also extra latency/cost regardless of model, and any LLM judge reintroduces a
# parse-failure mode. These two checks are deterministic, reuse already-loaded
# models, and cannot fail to parse.
import re

CONFIG_GATE = {
    # ponytail: CALIBRATION KNOB, not a derived constant. 0.50 is a starting guess.
    # The self-test below prints the real score distribution -- set this from the
    # scores your own grounded answers produce, don't trust the default.
    "min_sentence_similarity": 0.50,
    "min_sentence_chars"     : 25,     # shorter fragments are punctuation noise
    "block_on_fail"          : True,   # ungrounded answer -> replaced by refusal
}

REFUSAL = {
    "en": "I don't have enough information to answer this question.",
    "ar": "لا أملك معلومات كافية للإجابة على هذا السؤال.",
}

# Arabic-Indic and Persian digits -> ASCII, so "١٠ مغ" and "10 mg" compare equal.
_DIGIT_MAP = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")


def _numbers(text: str) -> set:
    """Every numeric literal in the text, digit-system normalized."""
    return set(re.findall(r"\d+(?:\.\d+)?", text.translate(_DIGIT_MAP)))


def _sentences(text: str) -> list:
    """Split on both Latin and Arabic terminators. Arabic '؟' is not '?'."""
    parts = re.split(r"[.!?؟\n]+", text)
    return [s.strip() for s in parts if len(s.strip()) >= CONFIG_GATE["min_sentence_chars"]]


def check_grounding(answer: str, context: str, language: str) -> dict:
    """
    Is every claim in `answer` supported by `context`?
    Returns a verdict dict; never raises -- an internal failure fails CLOSED.
    """
    # A correct refusal is the safe outcome, not an ungrounded claim. Without this
    # the gate would flag the model's own honesty as a hallucination.
    #
    # BUG FIXED: this used to be a plain substring test (REFUSAL in answer), which
    # matched an answer that merely QUOTES the refusal phrase mid-sentence inside a
    # much longer, substantive answer -- observed live: a real dosage answer (with
    # real numbers) that added "...it is 'I don't have enough information to answer
    # this question.' as the context provides dosages for specific conditions..."
    # The old check short-circuited to grounded=True on that substring match, which
    # skipped BOTH the numeric and semantic checks entirely for an answer containing
    # unverified numbers -- the fail-closed guarantee broke exactly where it matters.
    # Fix: the refusal phrase must account for (nearly) the WHOLE answer, not just
    # appear somewhere in it. ponytail: 1.5x is a calibration knob for minor
    # formatting slack (trailing space/punctuation), not a derived constant.
    _refusal = REFUSAL.get(language, "")
    if _refusal and _refusal.lower() in answer.lower() \
       and len(answer.strip()) <= len(_refusal) * 1.5:
        return {"grounded": True, "reason": "refusal", "min_similarity": 1.0,
                "ungrounded_sentences": [], "hallucinated_numbers": []}

    try:
        # ── check 1: numeric grounding ──
        # Highest-consequence failure mode in a medication-leaflet system is an
        # invented dose. Exact set membership, no similarity involved.
        bad_numbers = sorted(_numbers(answer) - _numbers(context))

        # ── check 2: semantic grounding ──
        # ponytail: KNOWN CEILING -- cosine measures topical overlap, not entailment.
        # "Take this with alcohol" vs context "Do NOT take this with alcohol" scores
        # very high and passes. Negation and reversed-polarity claims are NOT caught
        # by this check; only off-topic fabrication and invented numbers are. Closing
        # that needs a real NLI/entailment model (e.g. an mDeBERTa XNLI checkpoint)
        # scoring each answer sentence against its best-matching context sentence --
        # a second model on the GPU, hence not free. Do not describe this gate as a
        # faithfulness guarantee; it is a fabrication filter.
        ans_sents = _sentences(answer)
        ctx_sents = _sentences(context)

        # Bulleted answers -- which this model produces constantly in Arabic -- split
        # into fragments shorter than min_sentence_chars and get filtered to nothing.
        # The old code then returned min_similarity=1.0 and PASSED, so a list-shaped
        # hallucination skipped the semantic check entirely while reporting a perfect
        # score. Fall back to scoring the whole answer as one unit instead.
        if not ans_sents and answer.strip():
            ans_sents = [answer.strip()]
        if not ctx_sents and context.strip():
            ctx_sents = [context.strip()]

        if not ans_sents:
            # genuinely empty answer -- nothing to deliver, so fail closed rather
            # than reporting the vacuous "nothing unsupported was found" pass.
            return {"grounded": False, "reason": "empty answer",
                    "min_similarity": 0.0, "ungrounded_sentences": [],
                    "hallucinated_numbers": bad_numbers}
        if not ctx_sents:
            return {"grounded": False, "reason": "empty context", "min_similarity": 0.0,
                    "ungrounded_sentences": ans_sents, "hallucinated_numbers": bad_numbers}

        a_vecs = embed_model.encode(ans_sents, normalize_embeddings=True)
        c_vecs = embed_model.encode(ctx_sents, normalize_embeddings=True)
        sims   = a_vecs @ c_vecs.T          # normalized -> dot product IS cosine
        best   = sims.max(axis=1)

        thr        = CONFIG_GATE["min_sentence_similarity"]
        ungrounded = [(s, float(b)) for s, b in zip(ans_sents, best) if b < thr]

        # BUG FIXED (observed live, twice, across two runs): blocking on ANY single
        # ungrounded sentence let one purely structural sentence -- a trailing "I
        # don't have enough information" hedge, or a leading "According to the
        # sources:" framing line -- veto an otherwise accurate, well-grounded
        # multi-sentence answer. Neither sentence is a medical claim, so neither
        # should be able to single-handedly discard four correct bullet points.
        # Fix: block on a MAJORITY of sentences failing, not any one. Still fails
        # closed on genuine fabrication -- a single-sentence answer that's wrong is
        # still 100% ungrounded, still blocked; a multi-sentence answer where most
        # content is unsupported still trips this. ponytail: known ceiling -- exactly
        # half-ungrounded on a very short (e.g. 2-sentence) answer does NOT trigger
        # (needs strictly >50%), untested edge case, not yet observed in practice.
        majority_ungrounded = len(ungrounded) > len(ans_sents) / 2

        return {
            "grounded"            : (not majority_ungrounded) and (not bad_numbers),
            "reason"              : "ok" if (not majority_ungrounded and not bad_numbers) else "unsupported content",
            "min_similarity"      : float(best.min()),
            "ungrounded_sentences": ungrounded,
            "hallucinated_numbers": bad_numbers,
        }

    except Exception as e:
        # FAIL CLOSED. In offline eval a broken metric costs a data point; here it
        # would ship an unverified medical answer to a user. Never default to pass.
        logger.error(f"Grounding gate error: {e}")
        return {"grounded": False, "reason": f"gate error: {e}", "min_similarity": 0.0,
                "ungrounded_sentences": [], "hallucinated_numbers": []}


# ── self-check: the gate must catch an invented dose ──
_ctx = ("The recommended dose is 10 mg once daily. Store below 25 degrees Celsius. "
        "Common side effects include dizziness and a dry cough.")
_ok  = check_grounding("The recommended dose is 10 mg once daily.", _ctx, "en")
_bad = check_grounding("The recommended dose is 80 mg once daily.", _ctx, "en")
_ref = check_grounding(REFUSAL["en"], _ctx, "en")

assert _ok["grounded"],       f"grounded answer rejected: {_ok}"
assert not _bad["grounded"],  f"INVENTED DOSE PASSED THE GATE: {_bad}"
assert "80" in _bad["hallucinated_numbers"], f"wrong failure reason: {_bad}"
assert _ref["grounded"],      f"correct refusal flagged as hallucination: {_ref}"

# a bulleted answer must still be scored, not silently skipped with a perfect 1.000
_bullets = check_grounding("- headache\n- nausea\n- dizziness", _ctx, "en")
assert _bullets["min_similarity"] < 1.0, \
    f"bullet list bypassed the semantic check: {_bullets}"
_empty = check_grounding("", _ctx, "en")
assert not _empty["grounded"], f"empty answer passed: {_empty}"

# the exact bug seen live: a real answer that QUOTES the refusal mid-sentence must
# NOT be treated as a refusal -- its numbers still need checking.
_quoted = check_grounding(
    "The maximum dose is 200 mg/day. However, if asked generally, it is "
    "\"I don't have enough information to answer this question.\" since the "
    "context covers specific conditions only.",
    _ctx, "en",
)
assert _quoted["reason"] != "refusal", f"quoted refusal wrongly bypassed the gate: {_quoted}"
assert "200" in _quoted["hallucinated_numbers"], \
    f"embedded-refusal answer skipped the numeric check: {_quoted}"

# the exact pattern seen live twice: one purely structural sentence (a framing
# line or a trailing hedge) must not veto an otherwise well-grounded answer.
_framed = check_grounding(
    "According to the sources: "
    "The recommended dose is 10 mg once daily. "
    "Store below 25 degrees Celsius. "
    "Common side effects include dizziness and a dry cough.",
    _ctx, "en",
)
assert _framed["grounded"], \
    f"one structural sentence wrongly blocked an otherwise-grounded answer: {_framed}"
# but a genuinely fabricated single-sentence answer must still be blocked --
# the majority-vote fix must not weaken detection of real hallucination.
_single_bad = check_grounding("This medicine cures pancreatic cancer completely.", _ctx, "en")
assert not _single_bad["grounded"], f"fully fabricated answer passed: {_single_bad}"
print(f"✅ Groundedness gate self-check passed "
      f"(grounded sim={_ok['min_similarity']:.3f}, caught number {_bad['hallucinated_numbers']})")
print(f"   Calibrate CONFIG_GATE['min_sentence_similarity'] (now "
      f"{CONFIG_GATE['min_sentence_similarity']}) against the per-test scores below.")


# ════════════════════════════════════════════════════════════
# CELL 3 — Full RAG function (C4 + C5)
# ════════════════════════════════════════════════════════════
def rag_answer(query: str, document_id_filter: int | None = None) -> dict:
    """End-to-end RAG: retrieve → rerank → generate → groundedness gate."""
    pipeline_result = run_pipeline(query, document_id_filter=document_id_filter)
    context         = pipeline_result["context"]
    language        = pipeline_result["language"]
    generation      = generate_answer(query, context, language)

    # Gate judges against generation["context_used"] -- the truncated text the model
    # actually saw. Judging against the full context would credit the model for
    # content it was never shown.
    verdict = check_grounding(generation["answer"], generation["context_used"], language)

    answer = generation["answer"]
    if CONFIG_GATE["block_on_fail"] and not verdict["grounded"]:
        logger.warning(f"Gate BLOCKED answer: {verdict['reason']} | "
                       f"bad numbers={verdict['hallucinated_numbers']}")
        answer = REFUSAL.get(language, REFUSAL["en"])

    return {
        "query"          : query,
        "language"       : language,
        "answer"         : answer,
        "answer_raw"     : generation["answer"],   # pre-gate, kept for inspection
        "grounding"      : verdict,
        "retrieval_score": pipeline_result["retrieval_score"],
        "rewrite_count"  : pipeline_result["rewrite_count"],
        "reranked_chunks": pipeline_result["reranked_chunks"],
        "input_tokens"   : generation["input_tokens"],
        "output_tokens"  : generation["output_tokens"],
    }

def print_result(result: dict):
    print(f"Query          : {result['query']}")
    print(f"Language       : {result['language'].upper()}")
    print(f"Retrieval score: {result['retrieval_score']:.4f}")
    print(f"Input tokens   : {result['input_tokens']}")
    print(f"Output tokens  : {result['output_tokens']}")
    g = result["grounding"]
    flag = "PASS" if g["grounded"] else "BLOCKED"
    print(f"Groundedness   : {flag} | min sentence sim {g['min_similarity']:.3f} | {g['reason']}")
    if g["hallucinated_numbers"]:
        print(f"  numbers not in context : {g['hallucinated_numbers']}")
    for s, sc in g["ungrounded_sentences"][:3]:
        print(f"  unsupported ({sc:.3f}) : {s[:80]}")
    print(f"\n── Answer ──\n{result['answer']}")
    if result["answer"] != result["answer_raw"]:
        print(f"\n── Raw answer (gate-blocked) ──\n{result['answer_raw']}")

print("✅ RAG pipeline defined")

# ════════════════════════════════════════════════════════════
# CELL 4 — Tests
# ════════════════════════════════════════════════════════════

# ── Test 1: English side effects ──
print("\n" + "=" * 60)
print("TEST 1 — English: Side Effects")
print("=" * 60)
result_1 = rag_answer("What are the side effects of this medication?")
print_result(result_1)

# ── Test 2: Arabic side effects ──
print("\n" + "=" * 60)
print("TEST 2 — Arabic: Side Effects")
print("=" * 60)
result_2 = rag_answer("ما هي الآثار الجانبية لهذا الدواء؟")
print_result(result_2)

# ── Test 3: English dosage ──
print("\n" + "=" * 60)
print("TEST 3 — English: Dosage")
print("=" * 60)
result_3 = rag_answer("What is the recommended dosage for adults?")
print_result(result_3)

# ── Test 4: Arabic storage ──
print("\n" + "=" * 60)
print("TEST 4 — Arabic: Storage")
print("=" * 60)
result_4 = rag_answer("كيف يتم تخزين هذا الدواء؟")
print_result(result_4)

# ── Test 5: English pregnancy ──
print("\n" + "=" * 60)
print("TEST 5 — English: Pregnancy")
print("=" * 60)
result_5 = rag_answer("Is this medication safe during pregnancy?")
print_result(result_5)

# ════════════════════════════════════════════════════════════
# CELL 5 — Summary
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  COMPONENT 5 COMPLETE — LLM GENERATION READY")
print("=" * 60)
print(f"  ✅ Model       : {MODEL_NAME}")
print("  ✅ Quantization: INT4 BitsAndBytes NF4")
print("  ✅ Languages   : Arabic + English")
print("  ✅ Prompts     : Separate AR/EN medical templates")
print("  ✅ Safety      : Context-only prompting + runtime groundedness gate")
print("  ✅ Pipeline    : Component 4 + 5 end-to-end ✅")
print("=" * 60)
print("\n✅ Component 5 Complete")
print("   Ready for Component 6 — Evaluation")

In [ ]:
"""
================================================================
 Bilingual Arabic-English Medical RAG System
 Dataset : PEACH RAG Dataset (Patient Information Leaflets)
 Target  : ArabicNLP Workshop @ ACL/EMNLP
 Component 6 — Evaluation
 DeepEval + LLM-as-Judge + BERTScore (AR + EN)
================================================================
"""

# ════════════════════════════════════════════════════════════
# CELL 1 — Install dependencies
# ════════════════════════════════════════════════════════════
!pip install deepeval bert-score datasets nest_asyncio -q  # ragas dropped: ragas>=0.4 has a broken ChatVertexAI import (github.com/vibrantlabsai/ragas/issues/2741), ragas<0.4 pin did not resolve cleanly either

# ════════════════════════════════════════════════════════════
# CELL 2 — Imports
# ════════════════════════════════════════════════════════════
import json
import logging
import pandas as pd
import torch
import nest_asyncio
from bert_score import score as bert_score
from deepeval.models.base_model import DeepEvalBaseLLM
# ponytail: FaithfulnessMetric / ContextualRecallMetric / ContextualPrecisionMetric
# deliberately not imported -- all three require DeepEval's judge to produce a
# multi-item verdict-list JSON, which has now failed with BOTH models tried here:
# Falcon-H1 (faithfulness 0/12, context_recall 3/12) and Qwen2.5-7B (context_precision
# 0/12, uniformly "invalid JSON" -- even though the SAME run's simpler single-verdict
# AnswerRelevancyMetric succeeded 12/12). Counter-intuitively the larger, more capable
# Qwen model did WORSE on this specific task than the smaller Falcon-H1 did (which
# managed 10/12 after a token-budget fix) -- plausible explanation is Qwen wrapping
# JSON in more reasoning/preamble that breaks DeepEval's strict parser, not
# confirmed. See CLAUDE.md "Model swap" for the full evidence. Only AnswerRelevancy
# has proven reliable across both models.
from deepeval.metrics import (
    AnswerRelevancyMetric,
)
from deepeval.test_case import LLMTestCase

nest_asyncio.apply()  # Colab already runs an event loop; DeepEval needs this to run its own

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

# ════════════════════════════════════════════════════════════
# CELL 3 — Evaluation test set
# Hand-crafted QA pairs from PEACH dataset
# Both Arabic and English — covers all major categories
# ════════════════════════════════════════════════════════════
"""
Why hand-crafted test set?
- PEACH has no labeled QA pairs
- For workshop paper: 10-20 QA pairs is standard
- Covers: side effects, dosage, storage, warnings, pregnancy
- Both AR + EN — measures cross-lingual RAG performance
"""

# ponytail: EVAL_QA questions used to be generic ("this medication"), but the corpus
# has 464 different drugs -- retrieval for a generic question just grabbed whatever
# chunk was semantically closest across ANY drug, giving inconsistent grounding
# question to question and language to language. Each question now names a real
# document from the corpus (EN: Linopril/lisinopril, doc_id=502; AR: Logynon oral
# contraceptive, doc_id=498), retrieval is filtered to that document_id (see
# retrieve_chunks), and ground_truth is copied verbatim from that document's actual
# chunk_text -- not hand-typed generic drug-leaflet boilerplate.
EVAL_QA = [
    # ── English queries (Linopril / lisinopril, document_id=502) ──
    {
        "question"          : "What are the side effects of Linopril?",
        "ground_truth"      : "Common side effects (affecting 1 to 10 in 100 users) include headache, feeling dizzy or light-headed especially when standing up quickly, diarrhoea, a dry cough that does not go away, and being sick (vomiting).",
        "language"          : "en",
        "document_id"       : 502,
    },
    {
        "question"          : "What is the recommended dosage of Linopril for adults with high blood pressure?",
        "ground_truth"      : "For high blood pressure, the recommended starting dose is 10 mg once a day, and the usual long-term dose is 20 mg once a day.",
        "language"          : "en",
        "document_id"       : 502,
    },
    {
        "question"          : "Is Linopril safe to take during pregnancy?",
        "ground_truth"      : "Linopril is not recommended in early pregnancy and must not be taken if you are more than 3 months pregnant, as it may cause serious harm to your baby.",
        "language"          : "en",
        "document_id"       : 502,
    },
    {
        "question"          : "How should Linopril tablets be stored?",
        "ground_truth"      : "Keep out of the reach and sight of children. Do not use Linopril tablets after the expiry date. Store below 30°C.",
        "language"          : "en",
        "document_id"       : 502,
    },
    {
        "question"          : "What should I do if I forget to take a dose of Linopril?",
        "ground_truth"      : "If you forget to take a dose, take it as soon as you remember. However, if it is nearly time for the next dose, skip the missed dose. Do not take a double dose to make up for a forgotten dose.",
        "language"          : "en",
        "document_id"       : 502,
    },
    {
        "question"          : "Can Linopril be taken with food?",
        "ground_truth"      : "It does not matter if you take Linopril before or after food.",
        "language"          : "en",
        "document_id"       : 502,
    },
    {
        "question"          : "What are the contraindications of Linopril?",
        "ground_truth"      : "Do not take Linopril if you are allergic to lisinopril or any of its other ingredients, have ever had an allergic reaction (angioedema) to another ACE inhibitor, are more than 3 months pregnant, or are taking a blood pressure medicine containing aliskiren together with diabetes or kidney problems.",
        "language"          : "en",
        "document_id"       : 502,
    },
    # ── Arabic queries (Logynon oral contraceptive, document_id=498) ──
    {
        "question"          : "ما هي الآثار الجانبية للوجينون؟",
        "ground_truth"      : "الآثار الجانبية الشائعة للوجينون (قد تصل إلى 1 من كل 10 سيدات) تشمل تقلبات المزاج، مزاج مكتئب، صداع، غثيان، ألم في البطن، ألم أو وجع في الثدي، وزيادة الوزن.",
        "language"          : "ar",
        "document_id"       : 498,
    },
    {
        "question"          : "كيف يتم تخزين لوجينون؟",
        "ground_truth"      : "يحفظ لوجينون بعيدا عن متناول ونظر الأطفال، في درجة حرارة أقل من 30 درجة مئوية، ولا يستخدم بعد تاريخ انتهاء الصلاحية المذكور على العبوة.",
        "language"          : "ar",
        "document_id"       : 498,
    },
    {
        "question"          : "ما هي الجرعة الموصى بها من لوجينون؟",
        "ground_truth"      : "تؤخذ حبة واحدة من لوجينون يوميا لمدة 21 يوما في نفس الوقت تقريبا كل يوم، ثم تتوقف عن تناول الأقراص لمدة 7 أيام قبل بدء شريط جديد.",
        "language"          : "ar",
        "document_id"       : 498,
    },
    {
        "question"          : "هل لوجينون آمن أثناء الحمل؟",
        "ground_truth"      : "يجب عدم تناول لوجينون إذا كنت حاملا. إذا أصبحت حاملا أثناء تناوله، يجب التوقف عن تناوله فورا وزيارة الطبيب.",
        "language"          : "ar",
        "document_id"       : 498,
    },
    {
        "question"          : "ماذا أفعل إذا نسيت تناول قرص لوجينون؟",
        "ground_truth"      : "إذا تأخرت في تناول القرص لفترة أقل من 12 ساعة، فحماية منع الحمل لا تزال مضمونة وعليك تناول القرص المنسي في أسرع وقت ممكن. إذا تجاوز التأخير 12 ساعة، فقد لا تكون الحماية مضمونة وقد تحتاجين إلى استخدام وسيلة إضافية لمنع الحمل.",
        "language"          : "ar",
        "document_id"       : 498,
    },
]

logger.info(f"Evaluation test set: {len(EVAL_QA)} QA pairs")
logger.info(f"English: {sum(1 for q in EVAL_QA if q['language']=='en')}")
logger.info(f"Arabic : {sum(1 for q in EVAL_QA if q['language']=='ar')}")

# ════════════════════════════════════════════════════════════
# CELL 4 — Generate RAG answers for all test questions
# ════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("Generating RAG Answers for Evaluation Set")
logger.info("=" * 60)

eval_results = []

for i, qa in enumerate(EVAL_QA):
    logger.info(f"Processing {i+1}/{len(EVAL_QA)}: {qa['question'][:50]}")

    try:
        result = rag_answer(qa["question"], document_id_filter=qa["document_id"])

        eval_results.append({
            "question"        : qa["question"],
            "ground_truth"    : qa["ground_truth"],
            # "answer" is the DELIVERED answer -- post-gate, i.e. what a real user
            # would receive. If the gate blocked it this is the refusal string, and
            # BERTScore/DeepEval will score it low. That is correct end-to-end
            # measurement, but it means a metric drop is ambiguous on its own:
            # it could be worse generation OR the gate firing. "gate_blocked" below
            # disambiguates -- always read the two together.
            "answer"          : result["answer"],
            "answer_raw"      : result["answer_raw"],      # pre-gate generation
            "gate_blocked"    : not result["grounding"]["grounded"],
            "gate_reason"     : result["grounding"]["reason"],
            "gate_min_sim"    : result["grounding"]["min_similarity"],
            "gate_bad_numbers": result["grounding"]["hallucinated_numbers"],
            "language"        : qa["language"],
            "contexts"        : [c["text"] for c in result["reranked_chunks"]],
            "retrieval_score" : result["retrieval_score"],
            "rewrite_count"   : result["rewrite_count"],
            "input_tokens"    : result["input_tokens"],
            "output_tokens"   : result["output_tokens"],
        })

    except Exception as e:
        logger.error(f"Failed: {qa['question'][:40]} | {e}")
        continue

logger.info(f"✅ Generated {len(eval_results)} answers")

# ── Gate calibration: 12 more data points than Component 5's 5 smoke tests ──
# CONFIG_GATE["min_sentence_similarity"] is a guessed default. Set it from the
# distribution below, not from intuition: it belongs BELOW the lowest score among
# answers you judge correct, or the gate blocks good answers. If blocked and passed
# answers overlap in score, no single threshold separates them and the semantic half
# is the wrong instrument for those cases -- say so rather than tuning to fit.
_sims    = sorted(r["gate_min_sim"] for r in eval_results)
_blocked = [r for r in eval_results if r["gate_blocked"]]
print("\n" + "=" * 60)
print("  GROUNDEDNESS GATE — CALIBRATION")
print("=" * 60)
print(f"  Threshold in use : {CONFIG_GATE['min_sentence_similarity']}")
print(f"  Blocked          : {len(_blocked)}/{len(eval_results)}")
if _sims:
    print(f"  min sim range    : {_sims[0]:.3f} - {_sims[-1]:.3f} "
          f"(median {_sims[len(_sims)//2]:.3f})")
for r in _blocked:
    print(f"    BLOCKED [{r['language']}] sim={r['gate_min_sim']:.3f} "
          f"{r['gate_reason']} nums={r['gate_bad_numbers']} | {r['question'][:45]}")
if not _blocked:
    print("  Gate never fired -- it is not false-positiving, but this run gives")
    print("  NO evidence about its recall. Do not report it as validated.")
print("=" * 60)

# ════════════════════════════════════════════════════════════
# CELL 4b — Retrieval quality audit (BERTScore, no LLM judge)
# ════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("Retrieval Quality Audit")
logger.info("=" * 60)

"""
Why BERTScore instead of plain word overlap
- DeepEval's multi-item-verdict-list metrics (context_recall, context_precision,
  faithfulness) all go through the same local judge model, which has proven unable
  to reliably follow DeepEval's exact JSON schema for that shape of task with EITHER
  model tried in this project (see CLAUDE.md "Model swap"). A low DeepEval score
  doesn't tell us whether retrieval actually missed the right chunk, or whether the
  judge just failed to score it -- two different problems needing two different fixes.
- Plain word overlap is a crude lexical match: misses paraphrases/synonyms, and
  is weak for Arabic specifically -- which is exactly why BERTScore (not ROUGE)
  was already chosen elsewhere in this notebook for scoring answer quality
  ("handles morphological richness"). Reusing it here for retrieval sidesteps
  the same weakness while staying judge-independent (a fixed encoder score,
  not a generative LLM parsing free-form JSON).
- Scored per individual reranked chunk, taking the MAX across the top-5 --
  answers "did retrieval surface AT LEAST ONE chunk that matches the ground
  truth", which is what retrieval quality actually means. Scoring the whole
  concatenated 5-chunk context instead would dilute a genuinely good top chunk
  with 4 unrelated ones and understate retrieval quality.
"""

def score_chunks(pairs, lang, model_type):
    """pairs: list of (eval_results-index, chunk_text). Returns {index: max F1 across its chunks}."""
    if not pairs:
        return {}
    cands = [c for _, c in pairs]
    refs  = [eval_results[qi]["ground_truth"] for qi, _ in pairs]
    _, _, F1 = bert_score(cands, refs, lang=lang, model_type=model_type, verbose=False, device="cpu")
    best = {}
    for (qi, _), f1 in zip(pairs, F1):
        best[qi] = max(best.get(qi, 0.0), f1.item())
    return best

en_pairs = [(qi, c) for qi, r in enumerate(eval_results) if r["language"] == "en" for c in r["contexts"]]
ar_pairs = [(qi, c) for qi, r in enumerate(eval_results) if r["language"] == "ar" for c in r["contexts"]]

best_en = score_chunks(en_pairs, "en", "roberta-large")
best_ar = score_chunks(ar_pairs, "ar", "bert-base-multilingual-cased")

for qi, r in enumerate(eval_results):
    r["retrieval_bertscore_f1"] = best_en.get(qi, best_ar.get(qi, 0.0))
    logger.info(
        f"Retrieval audit | {r['language'].upper()} | "
        f"best_chunk_f1={r['retrieval_bertscore_f1']:.3f} | {r['question'][:50]}"
    )

avg_retrieval_f1 = sum(r["retrieval_bertscore_f1"] for r in eval_results) / len(eval_results)
print(f"\nMean best-chunk retrieval BERTScore F1: {avg_retrieval_f1:.3f}")
print("High (>0.6) -> retrieval surfaced the right chunk, DeepEval's judge is the bottleneck.")
print("Low  (<0.4) -> retrieval/reranking itself is missing the right chunk.")


# ════════════════════════════════════════════════════════════
# CELL 5 — BERTScore evaluation (AR + EN)
# ════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("BERTScore Evaluation")
logger.info("=" * 60)

"""
Why BERTScore?
- Measures semantic similarity between answer and ground truth
- Arabic: uses multilingual-e5 or mBERT
- English: uses roberta-large
- Better than ROUGE for Arabic (handles morphological richness)
- Standard metric in NLP papers
"""

en_results = [r for r in eval_results if r["language"] == "en"]
ar_results = [r for r in eval_results if r["language"] == "ar"]

# ── English BERTScore ──
if en_results:
    en_preds  = [r["answer"] for r in en_results]
    en_refs   = [r["ground_truth"] for r in en_results]

    P_en, R_en, F1_en = bert_score(
        en_preds, en_refs,
        lang       = "en",
        model_type = "roberta-large",
        verbose    = False,
        device     = "cpu",  # keep GPU free for the loaded model's generation calls below
    )

    avg_f1_en = F1_en.mean().item()
    logger.info(f"✅ EN BERTScore F1: {avg_f1_en:.4f}")

    for i, r in enumerate(en_results):
        r["bertscore_f1"] = F1_en[i].item()

# ── Arabic BERTScore ──
if ar_results:
    ar_preds = [r["answer"] for r in ar_results]
    ar_refs  = [r["ground_truth"] for r in ar_results]

    P_ar, R_ar, F1_ar = bert_score(
        ar_preds, ar_refs,
        lang       = "ar",
        model_type = "bert-base-multilingual-cased",
        verbose    = False,
        device     = "cpu",  # keep GPU free for the loaded model's generation calls below
    )

    avg_f1_ar = F1_ar.mean().item()
    logger.info(f"✅ AR BERTScore F1: {avg_f1_ar:.4f}")

    for i, r in enumerate(ar_results):
        r["bertscore_f1"] = F1_ar[i].item()

# ════════════════════════════════════════════════════════════
# CELL 6 — DeepEval evaluation (RAG metrics)
# ════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("DeepEval Evaluation")
logger.info("=" * 60)

"""
Why DeepEval instead of RAGAS?
- ragas>=0.4 ships a hard, unconditional import of ChatVertexAI from a
  langchain_community path that no longer exists (VertexAI support moved to
  the separate langchain-google-vertexai package) — this breaks `from ragas
  import evaluate` entirely, even though nothing here uses VertexAI.
  See github.com/vibrantlabsai/ragas/issues/2741. Pinning ragas<0.4 is the
  documented workaround but didn't resolve cleanly in this environment.
- DeepEval ships equivalent RAG metrics (faithfulness, answer relevancy,
  contextual precision, contextual recall) with no langchain/vertexai import
  chain, and wraps a local HF model directly.
- The loaded model (MODEL_NAME) is wrapped as DeepEval's judge model (DeepEvalBaseLLM subclass)
  — same on-premise, no-external-API judge used everywhere else in this
  notebook. Uses the same apply_chat_template + greedy-decode pattern as
  generate_answer (Component 5) — a raw tokenizer(prompt) call here would
  hit the same garbled-output bug that was fixed there.
"""

class LocalDeepEvalModel(DeepEvalBaseLLM):
    """Wraps the already-loaded model/tokenizer (MODEL_NAME, set in Setup) as
    DeepEval's judge -- a local, on-premise, no-external-API judge."""

    def load_model(self):
        return model

    def _run(self, prompt: str) -> str:
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            add_generation_prompt = True,
            tokenize               = True,
            return_dict             = True,
            return_tensors           = "pt",
            truncation                = True,
            max_length                 = 1024,  # naive SSM fallback scales ~O(seq_len^2); 2048 OOM'd
        ).to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                # History: this budget was tuned (384 -> 512) for context_precision's
                # multi-chunk verdict list under Falcon-H1, which got it to 10/12.
                # context_precision has since been dropped entirely -- under Qwen2.5-7B
                # it failed 0/12 with "invalid JSON" even at 512 tokens, so the failure
                # wasn't a token-budget problem for this model. 512 is kept as a
                # reasonable general ceiling for the one metric that remains
                # (answer_relevancy, a simpler single-verdict JSON task).
                max_new_tokens = 512,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )
        return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    def generate(self, prompt: str) -> str:
        return self._run(prompt)

    async def a_generate(self, prompt: str) -> str:
        return self._run(prompt)

    def get_model_name(self) -> str:
        return f"{MODEL_NAME} (local, on-premise judge)"

deepeval_llm = LocalDeepEvalModel()

deepeval_metrics = {
    "answer_relevancy"  : AnswerRelevancyMetric(model=deepeval_llm, include_reason=False),
}

# ponytail: one bad metric on one question shouldn't sink the whole eval run —
# a small local judge model occasionally produces output DeepEval can't parse.
deepeval_rows = []
for r in eval_results:
    test_case = LLMTestCase(
        input             = r["question"],
        actual_output     = r["answer"],
        expected_output   = r["ground_truth"],
        # ponytail: cap each chunk. Originally justified by Falcon-H1's naive SSM
        # fallback scaling ~O(seq_len^2) in memory (no causal-conv1d/mamba-ssm
        # installed) -- Qwen2.5 has no SSM layers, so that specific mechanism no
        # longer applies, but standard self-attention is also O(seq_len^2) in the
        # sequence dimension, so capping input size remains a reasonable general
        # safety margin, not something to remove on the architecture change alone.
        retrieval_context = [c[:500] for c in r["contexts"]],
    )
    row = {}
    for name, metric in deepeval_metrics.items():
        try:
            metric.measure(test_case)
            row[name] = metric.score
        except Exception as e:
            logger.warning(f"{name} failed for '{r['question'][:40]}': {e}")
            row[name] = None
    deepeval_rows.append(row)
    logger.info(f"DeepEval scores | {r['language'].upper()} | {row}")
    torch.cuda.empty_cache()  # DeepEval's per-question generate() calls fragment GPU memory fast

logger.info("✅ DeepEval evaluation complete")
ragas_df = pd.DataFrame(deepeval_rows)

# ════════════════════════════════════════════════════════════
# CELL 7 — LLM-as-Judge evaluation
# ════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("LLM-as-Judge Evaluation")
logger.info("=" * 60)

"""
Why LLM-as-Judge?
- Catches what automated metrics miss:
  medical accuracy, safety, coherence
- Uses the loaded model itself as judge (no external API needed)
- Scores 1-5 on: accuracy, safety, coherence
- Standard approach in recent RAG papers
"""

JUDGE_PROMPT = """You are a medical expert evaluating an AI assistant's answer.
Score the answer on a scale of 1-5 for each criterion.
Return ONLY a JSON object with scores.

Question: {question}
Reference Answer: {ground_truth}
AI Answer: {answer}

Criteria:
- accuracy  : Is the answer medically accurate? (1=wrong, 5=correct)
- safety    : Is the answer safe for patients? (1=unsafe, 5=safe)
- coherence : Is the answer clear and coherent? (1=unclear, 5=clear)

Return only: {{"accuracy": X, "safety": X, "coherence": X}}"""

judge_results = []

for r in eval_results:
    prompt = JUDGE_PROMPT.format(
        question     = r["question"],
        ground_truth = r["ground_truth"],
        answer       = r["answer"]
    )

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt = True,
        tokenize               = True,
        return_dict             = True,
        return_tensors           = "pt",
        truncation                = True,
        max_length                 = 1024,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 50,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Parse JSON scores
    try:
        # Find JSON in response
        start = response.find("{")
        end   = response.find("}") + 1
        if start != -1 and end != 0:
            scores = json.loads(response[start:end])
        else:
            scores = {"accuracy": 3, "safety": 3, "coherence": 3}
    except Exception:
        scores = {"accuracy": 3, "safety": 3, "coherence": 3}

    judge_results.append({
        "question" : r["question"],
        "language" : r["language"],
        **scores
    })

    logger.info(
        f"Judge scores | {r['language'].upper()} | "
        f"acc:{scores.get('accuracy',0)} "
        f"safe:{scores.get('safety',0)} "
        f"coh:{scores.get('coherence',0)}"
    )

# ════════════════════════════════════════════════════════════
# CELL 8 — Final metrics summary
# ════════════════════════════════════════════════════════════
judge_df = pd.DataFrame(judge_results)

en_judge = judge_df[judge_df.language == "en"]
ar_judge = judge_df[judge_df.language == "ar"]

print("\n" + "=" * 60)
print("  COMPONENT 6 — EVALUATION RESULTS")
print("=" * 60)

print("\n── BERTScore ──")
print(f"  English F1 : {avg_f1_en:.4f}")
print(f"  Arabic  F1 : {avg_f1_ar:.4f}")
print(f"  Overall F1 : {(avg_f1_en + avg_f1_ar) / 2:.4f}")

print("\n── DeepEval RAG Metrics ──")
# Each metric is a per-question try/except, so a mean can be computed over far fewer
# questions than were asked. Printing the mean alone invites comparing a 12-sample
# number against a 3-sample one as if they were equivalent. n is not optional here.
_n_total = len(ragas_df)
for _label, _col in [("Answer Relevancy", "answer_relevancy")]:
    _n = int(ragas_df[_col].count())
    _mean = ragas_df[_col].mean()
    _flag = ""
    if _n == 0:
        _flag = "  <- ALL judge calls failed; metric unusable"
    elif _n < _n_total * 0.75:
        _flag = f"  <- only {_n}/{_n_total} succeeded; DO NOT report this figure"
    _shown = "n/a   " if _n == 0 else f"{_mean:.4f}"
    print(f"  {_label:<18}: {_shown}  (n={_n}/{_n_total}){_flag}")

print("\n── LLM-as-Judge (1-5 scale) ──")
print(f"  EN Accuracy  : {en_judge['accuracy'].mean():.2f}")
print(f"  EN Safety    : {en_judge['safety'].mean():.2f}")
print(f"  EN Coherence : {en_judge['coherence'].mean():.2f}")
print(f"  AR Accuracy  : {ar_judge['accuracy'].mean():.2f}")
print(f"  AR Safety    : {ar_judge['safety'].mean():.2f}")
print(f"  AR Coherence : {ar_judge['coherence'].mean():.2f}")

print("\n── Per-language DeepEval ──")
en_idx = [i for i, r in enumerate(eval_results) if r["language"] == "en"]
ar_idx = [i for i, r in enumerate(eval_results) if r["language"] == "ar"]

en_ragas = ragas_df.iloc[en_idx]
ar_ragas = ragas_df.iloc[ar_idx]

print(f"  EN Answer Relevancy : {en_ragas['answer_relevancy'].mean():.4f}")
print(f"  AR Answer Relevancy : {ar_ragas['answer_relevancy'].mean():.4f}")

print("=" * 60)

# ════════════════════════════════════════════════════════════
# CELL 9 — Save all results
# ════════════════════════════════════════════════════════════
# Save detailed results
results_df = pd.DataFrame(eval_results)
results_df.to_csv("evaluation_results.csv", index=False)

# Save summary
summary = {
    "bertscore": {
        "english_f1": round(avg_f1_en, 4),
        "arabic_f1" : round(avg_f1_ar, 4),
        "overall_f1": round((avg_f1_en + avg_f1_ar) / 2, 4),
    },
    "deepeval": {
        "answer_relevancy"  : round(ragas_df["answer_relevancy"].mean(), 4),
        # faithfulness, context_recall, context_precision all omitted -- each
        # requires a multi-item verdict-list JSON that has failed with every local
        # judge model tried in this project (Falcon-H1 and Qwen2.5-7B alike).
        # See CLAUDE.md "Model swap" for the full history.
    },
    "llm_judge": {
        "en_accuracy" : round(en_judge["accuracy"].mean(), 2),
        "en_safety"   : round(en_judge["safety"].mean(), 2),
        "en_coherence": round(en_judge["coherence"].mean(), 2),
        "ar_accuracy" : round(ar_judge["accuracy"].mean(), 2),
        "ar_safety"   : round(ar_judge["safety"].mean(), 2),
        "ar_coherence": round(ar_judge["coherence"].mean(), 2),
    }
}

with open("evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n✅ Results saved:")
print("   evaluation_results.csv  — detailed per-question results")
print("   evaluation_summary.json — summary metrics")
print("\n✅ Component 6 Complete — All 6 Components Done!")
print("   System evaluated end-to-end")

In [ ]:
"""
================================================================
 Component 7 — Demo UI (Gradio)
================================================================
Wraps rag_answer() (Component 5) in a shareable web UI. Requires Setup,
Component 4, and Component 5 to have already been run in this session --
it calls their functions directly, nothing is reloaded here.

share=True tunnels through Gradio's own servers, so this works unmodified
on both Colab and Kaggle (Kaggle needs Internet: On, already required for
Setup's pip installs). The link is only live for as long as this notebook
session runs -- it is a live demo of this session, not a deployment. A
real deployment (FastAPI + Docker) is separate, later work.
"""

# ════════════════════════════════════════════════════════════
# CELL 1 — Install + UI
# ════════════════════════════════════════════════════════════
!pip install -q gradio

import gradio as gr

# A few real corpus questions (from Component 6's EVAL_QA) so the demo is
# clickable immediately, without the visitor having to know what's in the
# corpus. Asked unfiltered here (no document_id_filter) -- unlike the eval
# loop, this is a live query over the whole corpus, same as any real user.
EXAMPLE_QUERIES = [
    "What are the side effects of Linopril?",
    "What is the recommended dosage of Linopril for adults with high blood pressure?",
    "Is Linopril safe to take during pregnancy?",
    "ما هي الآثار الجانبية للوجينون؟",
    "كيف يتم تخزين لوجينون؟",
    "ما هي الجرعة الموصى بها من لوجينون؟",
]


def _grounding_badge(g: dict) -> str:
    if g["grounded"]:
        return f"✅ **Grounded** (min sentence similarity {g['min_similarity']:.2f})"
    return f"⚠️ **Blocked by groundedness gate** — {g['reason']} (min similarity {g['min_similarity']:.2f})"


def _sources_markdown(chunks: list) -> str:
    if not chunks:
        return "### Retrieved Sources\n\n_No sources retrieved._"
    lines = ["### Retrieved Sources"]
    for i, c in enumerate(chunks, 1):
        lines.append(
            f"**[{i}]** `{c['file_name']}` — {c['category']} ({c['language']}) "
            f"— rerank score {c.get('rerank_score', 0):.3f}\n\n"
            f"> {c['text'][:300].strip()}…"
        )
    return "\n\n".join(lines)


def ask(query: str):
    if not query or not query.strip():
        return "Enter a question first.", "", ""
    # document_id_filter intentionally omitted -- that's an eval-only knob
    # (see CLAUDE.md), a live demo query searches the whole corpus.
    result = rag_answer(query.strip())
    g = result["grounding"]
    meta = (
        f"**Language:** {result['language'].upper()} &nbsp;·&nbsp; "
        f"**Retrieval score:** {result['retrieval_score']:.3f} &nbsp;·&nbsp; "
        f"**Query rewrites:** {result['rewrite_count']} &nbsp;·&nbsp; "
        f"{_grounding_badge(g)}"
    )
    return result["answer"], meta, _sources_markdown(result["reranked_chunks"])


with gr.Blocks(title="GroundedRx — Bilingual Medical RAG") as demo:
    gr.Markdown(
        "# GroundedRx — Bilingual Medical RAG\n"
        "Ask a question in **English or Arabic** about a medication leaflet in the corpus. "
        "The answer is generated **only** from retrieved context, never the model's own "
        "knowledge, and passes through a runtime groundedness gate before being shown here — "
        "if the gate can't verify it, you get a refusal instead of a guess.\n\n"
        "_Demo only — not medical advice._"
    )
    query_box = gr.Textbox(
        label="Question",
        placeholder="e.g. What are the side effects of Linopril?",
        lines=2,
    )
    ask_btn = gr.Button("Ask", variant="primary")
    answer_box = gr.Textbox(label="Answer", lines=8, interactive=False)
    meta_box = gr.Markdown()
    sources_box = gr.Markdown()

    ask_btn.click(ask, inputs=query_box, outputs=[answer_box, meta_box, sources_box])
    query_box.submit(ask, inputs=query_box, outputs=[answer_box, meta_box, sources_box])
    gr.Examples(examples=EXAMPLE_QUERIES, inputs=query_box)

demo.launch(share=True, debug=False)

print("✅ Demo UI launched — use the public gradio.live link printed above.")
print("   Link dies when this session/runtime stops; re-run this cell to get a new one.")
